# Backtest Validation Notebook

This notebook runs correctness and edge-case validations on the CSV outputs produced by the backtest, plus a few direct data-quality checks against the raw input folders. Each section has a heading followed by code that runs that specific validation and prints clear PASS/FAIL results.

In [1]:
import os
import re
import math
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("../results")
RAW_DATA_DIR = Path("../allData")
UNDERLIERS = ["NIFTY", "BANKNIFTY"]
CSV_COLUMNS = ["Date", "Time", "Price", "Volume", "OpenInterest"]


def load_result_csv(path: Path):
    if not path.exists():
        print(f"MISSING FILE: {path}")
        return None
    df = pd.read_csv(path)
    if df.empty:
        print(f"EMPTY FILE: {path}")
    return df


def check(label, condition, detail=""):
    status = "PASS" if bool(condition) else "FAIL"
    print(f"[{status}] {label}" + (f" -- {detail}" if detail else ""))
    return bool(condition)


def parse_option_symbol(symbol: str):
    m = re.match(r"^([A-Z]+)(\d{6})(\d+)(CE|PE)$", symbol)
    if not m:
        return None
    underlier, expiry, strike, opt_type = m.groups()
    return {
        "underlier": underlier,
        "expiry": expiry,
        "strike": int(strike),
        "option_type": opt_type,
    }


def read_raw_tick_csv(path: Path):
    raw = pd.read_csv(path)
    if list(raw.columns[:5]) == CSV_COLUMNS:
        df = raw[CSV_COLUMNS].copy()
    else:
        df = pd.read_csv(path, header=None, names=CSV_COLUMNS)
    df["Date"] = df["Date"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["Time"] = df["Time"].astype(str)
    ts = pd.to_datetime(
        df["Date"].astype(str) + df["Time"].astype(str),
        format="%Y%m%d%H:%M:%S",
        errors="coerce",
    )
    df = df.loc[~ts.isna()].copy()
    ts = ts[~ts.isna()]
    df.insert(0, "Timestamp", ts)
    df = df.drop(columns=["Date", "Time"]).sort_values("Timestamp").reset_index(drop=True)
    return df


def find_case_insensitive_file(directory: Path, target_name: str):
    if not directory.exists():
        return None
    target = target_name.lower()
    for name in os.listdir(directory):
        if name.lower() == target:
            return directory / name
    return None


def list_trading_dates(raw_dir: Path):
    if not raw_dir.exists():
        return []
    dates = []
    for name in os.listdir(raw_dir):
        if name.startswith("NSE_"):
            dates.append(name.replace("NSE_", ""))
    return sorted(dates)


def load_all_results():
    out = {}
    for u in UNDERLIERS:
        out[u] = {
            "mtm": load_result_csv(RESULTS_DIR / f"{u}_mtm.csv"),
            "trades": load_result_csv(RESULTS_DIR / f"{u}_trades.csv"),
            "positions": load_result_csv(RESULTS_DIR / f"{u}_positions.csv"),
            "orders": load_result_csv(RESULTS_DIR / f"{u}_orders.csv"),
        }
    return out


results = load_all_results()


/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


## 1. Load all result files
Load MTM, trades, positions, and orders CSVs for each underlier and show their shapes.

In [2]:
for u in UNDERLIERS:
    for k, df in results[u].items():
        shape = None if df is None else df.shape
        print(f"{u} - {k}: {shape}")


NIFTY - mtm: (312825, 5)
NIFTY - trades: (8034, 9)
NIFTY - positions: (312825, 3)
NIFTY - orders: (16068, 6)
BANKNIFTY - mtm: (225957, 5)
BANKNIFTY - trades: (8600, 9)
BANKNIFTY - positions: (225957, 3)
BANKNIFTY - orders: (17200, 6)


## 2. Output timestamp integrity
Check that MTM timestamps are sorted and that there are no duplicate timestamps in the final MTM output.

In [3]:
for u in UNDERLIERS:
    mtm = results[u]["mtm"]
    if mtm is None or mtm.empty:
        continue
    ts = pd.to_datetime(mtm["timestamp"])
    check(f"{u}: timestamps sorted", ts.is_monotonic_increasing)
    dupes = int(ts.duplicated().sum())
    check(f"{u}: no duplicate timestamps", dupes == 0, f"{dupes} duplicates found")


[PASS] NIFTY: timestamps sorted
[PASS] NIFTY: no duplicate timestamps -- 0 duplicates found
[PASS] BANKNIFTY: timestamps sorted
[PASS] BANKNIFTY: no duplicate timestamps -- 0 duplicates found


## 3. Realized + Unrealized = Total PnL
At every timestamp, total PnL should equal realized PnL plus unrealized PnL.

In [4]:
for u in UNDERLIERS:
    mtm = results[u]["mtm"]
    if mtm is None or mtm.empty:
        continue
    lhs = mtm["realized_pnl"] + mtm["unrealized_pnl"]
    rhs = mtm["total_pnl"]
    max_diff = (lhs - rhs).abs().max()
    check(f"{u}: total_pnl == realized_pnl + unrealized_pnl", max_diff < 1e-6, f"max diff = {max_diff:.8f}")


[PASS] NIFTY: total_pnl == realized_pnl + unrealized_pnl -- max diff = 0.00000000
[PASS] BANKNIFTY: total_pnl == realized_pnl + unrealized_pnl -- max diff = 0.00000000


## 4. End-of-day flat positions
At the last timestamp of each day, there should be no open positions.

In [5]:
for u in UNDERLIERS:
    positions = results[u]["positions"]
    if positions is None or positions.empty:
        continue
    p = positions.copy()
    p["timestamp"] = pd.to_datetime(p["timestamp"])
    p["date"] = p["timestamp"].dt.date
    last_rows = p.groupby("date").tail(1)
    remaining = last_rows[last_rows["open_symbols"].fillna("").astype(str).str.strip() != ""]
    check(f"{u}: all days end flat", remaining.empty, f"{len(remaining)} day(s) with leftover positions")


[PASS] NIFTY: all days end flat -- 0 day(s) with leftover positions
[PASS] BANKNIFTY: all days end flat -- 0 day(s) with leftover positions


## 5. Max simultaneous open symbols
The strategy should not hold more than 2 symbols at once for an underlier (one CE and one PE).

In [6]:
for u in UNDERLIERS:
    positions = results[u]["positions"]
    if positions is None or positions.empty:
        continue
    open_counts = positions["open_symbols"].fillna("").apply(lambda s: 0 if str(s).strip() == "" else len(str(s).split(",")))
    max_open = int(open_counts.max()) if len(open_counts) else 0
    check(f"{u}: max simultaneous open symbols <= 2", max_open <= 2, f"observed max = {max_open}")


[PASS] NIFTY: max simultaneous open symbols <= 2 -- observed max = 2
[PASS] BANKNIFTY: max simultaneous open symbols <= 2 -- observed max = 2


## 6. Trade log arithmetic
For each trade, recompute realized PnL from entry price, exit price, quantity, and direction.

In [7]:
for u in UNDERLIERS:
    trades = results[u]["trades"]
    if trades is None or trades.empty:
        continue
    t = trades.copy()
    direction = t["direction"].astype(str).str.upper()
    sign = np.where(direction == "LONG", 1, -1)
    recomputed = (t["exit_price"] - t["entry_price"]) * t["quantity"] * sign
    max_diff = (recomputed - t["realized_pnl"]).abs().max()
    check(f"{u}: trade log PnL matches recomputed PnL", max_diff < 1e-6, f"max diff = {max_diff:.8f}")


[PASS] NIFTY: trade log PnL matches recomputed PnL -- max diff = 0.00000000
[PASS] BANKNIFTY: trade log PnL matches recomputed PnL -- max diff = 0.00000000


## 7. Sum of trade PnLs equals final realized PnL
The sum of trade-level realized PnL should equal the final realized PnL in the MTM series.

In [8]:
for u in UNDERLIERS:
    trades = results[u]["trades"]
    mtm = results[u]["mtm"]
    if trades is None or trades.empty or mtm is None or mtm.empty:
        continue
    sum_trade_pnl = trades["realized_pnl"].sum()
    final_realized = mtm["realized_pnl"].iloc[-1]
    diff = abs(sum_trade_pnl - final_realized)
    check(f"{u}: sum(trade PnL) == final realized_pnl", diff < 1e-3, f"diff = {diff:.6f}")


[PASS] NIFTY: sum(trade PnL) == final realized_pnl -- diff = 0.000000
[PASS] BANKNIFTY: sum(trade PnL) == final realized_pnl -- diff = 0.000000


## 8. Daily PnL reconciles with cumulative PnL
The sum of daily incremental PnL should equal the final cumulative total PnL.

In [9]:
for u in UNDERLIERS:
    mtm = results[u]["mtm"]
    if mtm is None or mtm.empty:
        continue
    m = mtm.copy()
    m["timestamp"] = pd.to_datetime(m["timestamp"])
    m["date"] = m["timestamp"].dt.date
    day_end = m.groupby("date", as_index=False)["total_pnl"].last()
    day_end["daily_pnl"] = day_end["total_pnl"].diff().fillna(day_end["total_pnl"])
    total_daily = day_end["daily_pnl"].sum()
    final_total = m["total_pnl"].iloc[-1]
    diff = abs(total_daily - final_total)
    check(f"{u}: sum(daily PnL) == final total PnL", diff < 1e-3, f"diff = {diff:.6f}")


[PASS] NIFTY: sum(daily PnL) == final total PnL -- diff = 0.000000
[PASS] BANKNIFTY: sum(daily PnL) == final total PnL -- diff = 0.000000


## 9. Drawdown sign check
Drawdown computed as total PnL minus running peak should never be positive.

In [10]:
for u in UNDERLIERS:
    mtm = results[u]["mtm"]
    if mtm is None or mtm.empty:
        continue
    running_peak = mtm["total_pnl"].cummax()
    drawdown = mtm["total_pnl"] - running_peak
    max_val = drawdown.max()
    check(f"{u}: drawdown always <= 0", max_val <= 1e-9, f"max drawdown value observed = {max_val:.8f}")


[PASS] NIFTY: drawdown always <= 0 -- max drawdown value observed = 0.00000000
[PASS] BANKNIFTY: drawdown always <= 0 -- max drawdown value observed = 0.00000000


## 10. Single expiry per day in executed trades
For a given underlier and day, the executed trades should come from only one expiry series.

In [11]:
for u in UNDERLIERS:
    trades = results[u]["trades"]
    if trades is None or trades.empty:
        continue
    t = trades.copy()
    t["entry_time"] = pd.to_datetime(t["entry_time"])
    t["date"] = t["entry_time"].dt.date
    t["expiry"] = t["symbol"].apply(lambda s: parse_option_symbol(str(s))["expiry"] if parse_option_symbol(str(s)) else None)
    per_day = t.groupby("date")["expiry"].nunique(dropna=True)
    bad_days = per_day[per_day > 1]
    check(f"{u}: one expiry per day in trade log", bad_days.empty, f"{len(bad_days)} day(s) traded multiple expiries")


[PASS] NIFTY: one expiry per day in trade log -- 0 day(s) traded multiple expiries
[PASS] BANKNIFTY: one expiry per day in trade log -- 0 day(s) traded multiple expiries


## 11. Raw futures duplicate-second check
Raw futures files can contain multiple rows in the same second. This check reports whether duplicate timestamps exist in the raw data and confirms that your loader should collapse them before strategy simulation.

In [12]:
dates = list_trading_dates(RAW_DATA_DIR)
if not dates:
    print(f"No raw dates found under {RAW_DATA_DIR}")
else:
    for u in UNDERLIERS:
        total_dupes = 0
        files_seen = 0
        for date in dates:
            fpath = find_case_insensitive_file(RAW_DATA_DIR / f"NSE_{date}" / "Futures (Continuous)", f"{u}-I.csv")
            if fpath is None:
                continue
            raw = read_raw_tick_csv(fpath)
            dupes = int(raw["Timestamp"].duplicated().sum())
            total_dupes += dupes
            files_seen += 1
        print(f"{u}: checked {files_seen} futures file(s), raw duplicate-second rows = {total_dupes}")
        check(f"{u}: raw duplicate-second detection ran", files_seen > 0)


NIFTY: checked 21 futures file(s), raw duplicate-second rows = 12731
[PASS] NIFTY: raw duplicate-second detection ran
BANKNIFTY: checked 21 futures file(s), raw duplicate-second rows = 150
[PASS] BANKNIFTY: raw duplicate-second detection ran


## 12. Missing selected-contract quote handling
Check whether any executed trade uses an entry or exit price that is missing or non-positive. This is a practical proxy to ensure the engine did not execute on invalid quotes.

In [13]:
for u in UNDERLIERS:
    trades = results[u]["trades"]
    if trades is None or trades.empty:
        continue
    bad_entry = trades["entry_price"].isna().sum() + (trades["entry_price"] <= 0).sum()
    bad_exit = trades["exit_price"].isna().sum() + (trades["exit_price"] <= 0).sum()
    total_bad = int(bad_entry + bad_exit)
    check(f"{u}: no invalid executed entry/exit prices", total_bad == 0, f"invalid price fields = {total_bad}")


[PASS] NIFTY: no invalid executed entry/exit prices -- invalid price fields = 0
[PASS] BANKNIFTY: no invalid executed entry/exit prices -- invalid price fields = 0


## 13. Deterministic strike tie-break / strike grid sanity
Check that all traded strikes belong to the available strike grid and that strikes are numeric and consistent. This does not prove tie-break correctness by itself, but it confirms symbol parsing and strike extraction behave as expected on executed trades.

In [14]:
for u in UNDERLIERS:
    trades = results[u]["trades"]
    if trades is None or trades.empty:
        continue
    parsed = trades["symbol"].astype(str).apply(parse_option_symbol)
    bad_parse = parsed.isna().sum()
    strikes = [p["strike"] for p in parsed if p is not None]
    strike_numeric = all(isinstance(s, int) for s in strikes)
    strike_positive = all(s > 0 for s in strikes)
    check(f"{u}: all traded symbols parse correctly", bad_parse == 0, f"bad parses = {bad_parse}")
    check(f"{u}: traded strikes are numeric", strike_numeric)
    check(f"{u}: traded strikes are positive", strike_positive)


[PASS] NIFTY: all traded symbols parse correctly -- bad parses = 0
[PASS] NIFTY: traded strikes are numeric
[PASS] NIFTY: traded strikes are positive
[PASS] BANKNIFTY: all traded symbols parse correctly -- bad parses = 0
[PASS] BANKNIFTY: traded strikes are numeric
[PASS] BANKNIFTY: traded strikes are positive


## 14. Missing or empty day resilience
Compare raw trading dates to result dates and report whether any raw dates produced no MTM rows. This helps detect silent crashes or skipped days.

In [15]:
raw_dates = set(list_trading_dates(RAW_DATA_DIR))
for u in UNDERLIERS:
    mtm = results[u]["mtm"]
    if mtm is None or mtm.empty:
        check(f"{u}: result contains MTM rows", False)
        continue
    m = mtm.copy()
    m["timestamp"] = pd.to_datetime(m["timestamp"])
    result_dates = set(m["timestamp"].dt.strftime("%Y%m%d").unique())
    missing_dates = sorted(raw_dates - result_dates)
    print(f"{u}: raw dates = {len(raw_dates)}, result dates = {len(result_dates)}")
    check(f"{u}: every raw date produced MTM rows", len(missing_dates) == 0, f"missing result dates = {missing_dates[:10]}")


NIFTY: raw dates = 21, result dates = 21
[PASS] NIFTY: every raw date produced MTM rows -- missing result dates = []
BANKNIFTY: raw dates = 21, result dates = 21
[PASS] BANKNIFTY: every raw date produced MTM rows -- missing result dates = []
